In [1]:
import numpy as np
from melt_estimation import (
    JointMeltEstimator,
    MELTSState,
    MeltConductivityModel,
    ModifiedArchieModel,
    ElasticAggregate,
    SeismicMeltReduction,
    SeismicDEM,
)

In [2]:
# melts_state, models = build_models_from_alphamelts("/path/to/run", T_target_C=1200, P_target_kbar=10)

melts_state = MELTSState(
    900, # temperature_K: float
    .5,  # pressure_GPa: float
    .3,  # melt_fraction: float            # Thermodynamic φ from MELTS (can be used as a prior)
    {},  # melt_composition: Dict[str, float]  # e.g., oxides wt% + H2O, CO2
    {},  # mineral_modes: Dict[str, float]     # e.g., {"olivine":0.55,"opx":0.25,...}
    2.5, # melt_density_kg_m3: Optional[float] = None
    2.7,  # solid_density_kg_m3: Optional[float] = None
)



In [3]:
conductivity_model = MeltConductivityModel(
    weight_percent_water=.1,
    temperature=1275,
    pressure=.2
)
print(f"sigma_melt = {conductivity_model.compute():0.4f} S/m")


sigma_melt = 1.1692 S/m


## Seismic Properties

| Rock Type | Bulk Modulus K (GPa) | Shear Modulus μ (GPa) | Estimated Density (g/cm³) | Source(s) |
|----------|-----------------------|------------------------|----------------------------|-----------|
| Granite  | 22–58 | ~24 (typical), range 20–30 | **2.54–2.66** | Granite engineering/materials tables and stone datasets (bulk modulus range; typical shear modulus ≈24 GPa; density 2.54–2.66 g/cm³). [1](https://www.nist.gov/publications/elastic-moduli-material-containing-composite-inclusions-effective-medium-theory-and)[2](https://github.com/mostafabbasi/RockPhysics_KeyLiteratures/blob/main/1974_OConnell%20and%20Budiansky.pdf)[3](https://www.eri.u-tokyo.ac.jp/KOHO/STAFF2/eng/ytakei_e.html) |
| Dacite   | 40–60 (derived from typical Vp, Vs, ρ) | 22–30 (derived) | **~2.40–2.60** | Values commonly **derived** via K=ρ(Vp²−4/3 Vs²) and μ=ρ Vs² using crustal igneous rock velocity/density compilations; dacitic lavas typically fall near 2.4–2.6 g/cm³. [4](https://books.google.com/books/about/The_Rock_Physics_Handbook.html?id=FMS-DwAAQBAJ)[5](https://cseg.ca/petrophysical-models-for-the-seismic-velocity-of-cracked-media/) |
| Basalt   | 55–85 (saturated, confining pressure) | 30–45 (saturated, confining pressure) | **~2.55–2.90** | Lab studies on DSDP basalts report moduli from measured Vp, Vs, ρ across pressures; example densities span ~2.55–2.73 g/cm³ in altered/vesicular samples and commonly extend toward ~2.9 g/cm³ in massive basalts. [6](https://www.academia.edu/66135453/Rock_Physics_Models_for_the_Seismic_Velocity_of_Cracked_Media)[7](https://earthref.org/ERR/70873/) |

In [3]:
seismic_model = SeismicDEM()

vp, vs = seismic_model.compute(2.6, .2, 50, 24, 15, 0)

print(f"Vs = {vs} m/s")
print(f"Vp = {vp} m/s")

Vs = 2.457265247363675 m/s
Vp = 4.797773785571498 m/s


In [ ]:
import matplotlib.pyplot as plt
# make a figure of melt percent

percent = np.linspace(0.01, 1, 100)

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)




In [11]:

estimator = JointMeltEstimator(melt_cond, archie, elastic, seismic)

res = estimator.invert_phi(target_rho_bulk_Ohm_m=5.0, target_vp_ms=7000.0,
                           T_K=melts_state.temperature_K,
                           comp=melts_state.melt_composition,
                           phi_bounds=(0.0, 0.20))
for key, value in res.items():
    print(f"{key} = {value}")

phi = 0.1265
sigma_melt = 5.434511700157324e-08
sigma_bulk = 0.008735003872412338
rho_bulk_Ohm_m = 114.48191833758523
vp_ms = 7003.780991851345
vs_ms = 3561.182770832287
misfit = 12000.586342257617


In [ ]:
# Now add volatile sensitivity that matters
model = MeltConductivityModel(
    weight_percent_water=.1,
    temperature=775,
    pressure=.2
)

# Evaluate at your MELTS-derived T and composition
print("sigma_melt =", model.compute())

2717514.287724697 116549.05735060386
sigma_melt = 0.0156616582873678
